# Experiment analysis — round-based ledger

Reads `results.tsv`, which this project writes with **10 columns** (the upstream
autoresearch notebook assumed 5 and will not parse it):

`exp_id  round  commit  val_bpb  sigma_mult  memory_gb  tok_per_s  epoch  status  description`

Two things this notebook does that the upstream one could not, and which matter once
there is more than one round:

- **Per-round baselines.** Upstream treats row 0 as *the* baseline. Across three rounds
  at different data scales and time budgets, `val_bpb` values from different rounds are
  not comparable at all — a Round 3 run has 7.5× the training budget of a Round 1 run.
  Every comparison here is scoped to its own round.
- **The noise band.** `sigma_mult` is the number each keep/discard was actually decided
  on (see `research-program.md` §4). A win at −1.2σ and a win at −6σ are different
  kinds of result, and plotting raw `val_bpb` hides that.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

COLS = ["exp_id", "round", "commit", "val_bpb", "sigma_mult",
        "memory_gb", "tok_per_s", "epoch", "status", "description"]

df = pd.read_csv("results.tsv", sep="\t")
assert list(df.columns) == COLS, (
    f"unexpected schema: {list(df.columns)}\n"
    f"expected: {COLS}\n"
    "If this is an upstream 5-column results.tsv, it is from a different protocol - "
    "start a fresh ledger rather than merging them."
)

for c in ["val_bpb", "sigma_mult", "memory_gb", "tok_per_s", "epoch"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["status"] = df["status"].str.strip().str.lower()
df["round"] = pd.to_numeric(df["round"], errors="coerce").astype("Int64")

print(f"{len(df)} experiments across rounds {sorted(df['round'].dropna().unique().tolist())}")
if len(df) == 0:
    print("\nLedger is empty - nothing has run yet. The rest of this notebook needs data.")
df.head(10)

## Validity gate first

Two checks that decide whether the numbers below mean anything. Run these before
reading any result, not after.

- **`epoch < 1.0`** — a run that looped its data measured memorization, not the change
  under test (`research-program.md` §1.2). Its verdict does not transfer to the next
  round, whatever its `val_bpb`.
- **`|sigma_mult|` sanity** — a keep recorded below 2σ means the decision rule was
  bypassed somewhere.

In [ ]:
bad_epoch = df[df["epoch"] >= 1.0]
if len(bad_epoch):
    print(f"INVALID — {len(bad_epoch)} run(s) exceeded 1.0 epoch. Raise SHARD_BUDGET and rerun:")
    print(bad_epoch[["exp_id", "round", "epoch", "val_bpb", "status"]].to_string(index=False))
else:
    print(f"epoch gate: OK (max {df['epoch'].max():.3f})" if len(df) else "epoch gate: no data")

bypassed = df[(df["status"] == "keep") & (df["sigma_mult"] > -2.0)]
if len(bypassed):
    print(f"\nCheck these {len(bypassed)} keep(s) under 2 sigma - legitimate only if the")
    print("change SIMPLIFIED the code (the section 4 simplification win):")
    print(bypassed[["exp_id", "round", "sigma_mult", "description"]].to_string(index=False))
else:
    print("\nsigma gate: no keeps under 2 sigma")

## Per-round summary

Hit rate is the honest headline number for a research loop: it says how often an idea
survived contact with the noise band. A high keep rate with most keeps clustered just
past 2σ usually means σ is underestimated, not that the ideas were good.

In [ ]:
rows = []
for r, g in df.groupby("round"):
    decided = g[g["status"].isin(["keep", "discard"])]
    keeps = g[g["status"] == "keep"]
    rows.append({
        "round": r,
        "runs": len(g),
        "keep": len(keeps),
        "discard": (g["status"] == "discard").sum(),
        "retest": (g["status"] == "retest").sum(),
        "crash": (g["status"] == "crash").sum(),
        "keep_rate": len(keeps) / len(decided) if len(decided) else np.nan,
        "baseline_bpb": g["val_bpb"].iloc[0] if len(g) else np.nan,
        "best_bpb": g.loc[g["status"] != "crash", "val_bpb"].min() if len(g) else np.nan,
        "median_sigma_of_keeps": keeps["sigma_mult"].median() if len(keeps) else np.nan,
    })
summary = pd.DataFrame(rows)
if len(summary):
    summary["round_delta"] = summary["baseline_bpb"] - summary["best_bpb"]
print(summary.to_string(index=False))

## Progress within each round

One panel per round, because `val_bpb` is only comparable inside a round. The running
minimum over kept runs is the frontier the branch actually advanced along.

In [ ]:
rounds = sorted(df["round"].dropna().unique().tolist())
if rounds:
    fig, axes = plt.subplots(1, len(rounds), figsize=(7 * len(rounds), 5), squeeze=False)
    for ax, r in zip(axes[0], rounds):
        g = df[df["round"] == r].reset_index(drop=True)
        live = g[g["status"] != "crash"]
        disc = live[live["status"] == "discard"]
        keep = live[live["status"] == "keep"]
        rt = live[live["status"] == "retest"]

        ax.scatter(disc.index, disc["val_bpb"], c="#cccccc", s=18, label="discard", zorder=2)
        ax.scatter(rt.index, rt["val_bpb"], c="#f39c12", s=34, label="retest", zorder=3)
        ax.scatter(keep.index, keep["val_bpb"], c="#2ecc71", s=58, edgecolors="black",
                   linewidths=0.5, label="keep", zorder=4)
        if len(keep):
            ax.step(keep.index, keep["val_bpb"].cummin(), where="post",
                    color="#27ae60", lw=2, alpha=0.8, label="running best", zorder=3)
        if len(g):
            base = g["val_bpb"].iloc[0]
            ax.axhline(base, color="#888", ls="--", lw=1, label="round baseline")
        for i, row in keep.iterrows():
            d = str(row["description"])[:40]
            ax.annotate(d, (i, row["val_bpb"]), textcoords="offset points",
                        xytext=(6, 6), fontsize=7.5, color="#1a7a3a")
        ax.set_title(f"Round {r} — {len(g)} runs")
        ax.set_xlabel("experiment # within round")
        ax.set_ylabel("val_bpb (lower is better)")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print("no rounds in the ledger yet")

## Effect sizes in sigma

This is the plot that separates signal from luck. The band between −2σ and +1σ is where
a result is indistinguishable from a reseed; anything decided inside it should either be
a simplification win or should not have been kept.

In [ ]:
live = df[df["status"] != "crash"].copy()
if len(live):
    fig, ax = plt.subplots(figsize=(11, max(3, 0.32 * len(live))))
    colors = {"keep": "#2ecc71", "discard": "#cccccc", "retest": "#f39c12"}
    y = np.arange(len(live))
    ax.barh(y, live["sigma_mult"], color=[colors.get(s, "#999") for s in live["status"]])
    ax.axvspan(-2, 1, color="#e74c3c", alpha=0.08)
    ax.axvline(-2, color="#e74c3c", ls="--", lw=1.2, label="keep threshold (-2 sigma)")
    ax.axvline(0, color="#333", lw=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels([f"r{r} {e}" for r, e in zip(live["round"], live["exp_id"])], fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel("effect size (sigma from round baseline)  <- better")
    ax.set_title("Shaded band = indistinguishable from seed noise")
    ax.legend(fontsize=8)
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    plt.show()
else:
    print("nothing to plot yet")

## What held, per round

The kept changes, in order, with the effect size each was decided on. This table is what
section 2 of a round assessment is built from.

In [ ]:
for r in rounds:
    keep = df[(df["round"] == r) & (df["status"] == "keep")]
    g = df[df["round"] == r]
    if not len(keep):
        print(f"Round {r}: no keeps\n")
        continue
    base, best = g["val_bpb"].iloc[0], keep["val_bpb"].min()
    print(f"Round {r}: {len(keep)} keeps, baseline {base:.6f} -> best {best:.6f} "
          f"(delta {base - best:+.6f})")
    for _, row in keep.iterrows():
        print(f"   {row['exp_id']:10s} {row['val_bpb']:.6f}  {row['sigma_mult']:+6.2f} sigma  "
              f"{row['description']}")
    # Individual deltas rarely sum to the cumulative delta. The gap IS the interaction
    # term, and naming it is the most useful line in a round assessment.
    print()

## Cost side

A val_bpb win that doubles VRAM or halves throughput is not free. Section 7's simplicity
criterion gets decided partly here.

In [ ]:
if len(live):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for r in rounds:
        g = live[live["round"] == r]
        axes[0].scatter(g["memory_gb"], g["val_bpb"], s=30, alpha=0.75, label=f"round {r}")
        axes[1].scatter(g["tok_per_s"], g["val_bpb"], s=30, alpha=0.75, label=f"round {r}")
    axes[0].set_xlabel("peak memory (GB)"); axes[0].set_ylabel("val_bpb")
    axes[1].set_xlabel("tokens / s"); axes[1].set_ylabel("val_bpb")
    for a in axes:
        a.grid(alpha=0.3); a.legend(fontsize=8)
    fig.suptitle("Cost vs result — points far left/right at equal bpb are the free wins")
    fig.tight_layout()
    plt.show()

## Crashes

Worth reading rather than skipping: a cluster of crashes in one direction usually means
that direction was never actually tested, and should be recorded as *inconclusive*
rather than refuted in the round assessment.

In [ ]:
crashes = df[df["status"] == "crash"]
if len(crashes):
    print(f"{len(crashes)} crash(es):")
    for _, row in crashes.iterrows():
        print(f"  r{row['round']} {row['exp_id']:10s} {row['description']}")
else:
    print("no crashes")